In [4]:
import numpy as np

np.random.seed(42)

def relu(z):
    return np.maximum(0,z)

def relu_derivative(z):
    return (z>0).astype(float)                # return 1 if z>0

def softmax(z):
    exp_z = np.exp(z - z.max(axis=1, keepdims= True))
    return exp_z / exp_z.sum(axis=1, keepdims=True)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

In [6]:
# MNIST: 784 inputs → 128 hidden → 64 hidden → 10 outputs

class NeuralNetwork:

    def __init__(self, layer_sizes):
        """
        layer_sizes: e.g. [784, 128, 64, 10]
        """
        self.layers = layer_sizes
        self.weights = []
        self.biases  = []


        for i in range(len(layer_sizes) - 1):
            n_in  = layer_sizes[i]
            n_out = layer_sizes[i + 1]
            W = np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)
            b = np.zeros((1, n_out))
            self.weights.append(W)
            self.biases.append(b)

        print(f"Network: {' → '.join(str(s) for s in layer_sizes)}")
        total_params = sum(W.size + b.size
                          for W, b in zip(self.weights, self.biases))
        print(f"Total parameters: {total_params:,}")
        
            
    def forward(self, X):
        """
        X shape: (batch_size, 784)
        Returns: output probabilities, shape (batch_size, 10)
        """
        self.activations = [X]   # store for backprop later
        self.z_values    = []

        current = X

        # Hidden layers — ReLU activation
        for i in range(len(self.weights) - 1):
            z = current @ self.weights[i] + self.biases[i]
            a = relu(z)
            self.z_values.append(z)
            self.activations.append(a)
            current = a
        
        z_out = current @ self.weights[-1] + self.biases[-1]
        a_out = softmax(z_out)
        self.z_values.append(z_out)
        self.activations.append(a_out)

        return a_out

    def loss(self, y_pred, y_true):
        """
        Cross-entropy loss for multi-class classification
        y_true: one-hot encoded, shape (batch_size, 10)
        y_pred: softmax output, shape (batch_size, 10)
        """
        n = y_true.shape[0]
        return -np.sum(y_true * np.log(y_pred + 1e-8)) / n        

In [7]:
#  Testing the Forward Pass

#  Creating a Network
nn = NeuralNetwork([784,128,64,10])

X_fake = np.random.randn(5,784)

output = nn.forward(X_fake)

print(f"\nInput shape:  {X_fake.shape}")
print(f"Output shape: {output.shape}")
print(f"\nPrediction probabilities for image 0:")
for digit, prob in enumerate(output[0]):
    bar = '█' * int(prob * 50)
    print(f"  Digit {digit}: {prob:.4f}  {bar}")

print(f"\nPredicted digit: {output[0].argmax()}")
print(f"Sum of probs:    {output[0].sum():.6f}")

Network: 784 → 128 → 64 → 10
Total parameters: 109,386

Input shape:  (5, 784)
Output shape: (5, 10)

Prediction probabilities for image 0:
  Digit 0: 0.0341  █
  Digit 1: 0.0912  ████
  Digit 2: 0.0028  
  Digit 3: 0.1504  ███████
  Digit 4: 0.0659  ███
  Digit 5: 0.1974  █████████
  Digit 6: 0.0522  ██
  Digit 7: 0.2473  ████████████
  Digit 8: 0.1454  ███████
  Digit 9: 0.0133  

Predicted digit: 7
Sum of probs:    1.000000


In [9]:
#  Some Exercises related to this model

# Question 1: how many parameters does a deeper network have?
nn_deep = NeuralNetwork([784, 256, 128, 64, 10])

# Question 2: what happens to output shape with different batch sizes?
X_batch_32 = np.random.rand(32, 784)
out = nn_deep.forward(X_batch_32)
print(out.shape)

# Question 3: what does relu do to negative values?
test = np.array([-3, -1, 0, 1, 3])
print(relu(test))   # predict before running

# Question 4: verify softmax sums to 1 for every sample in batch
print(out.sum(axis=1))

Network: 784 → 256 → 128 → 64 → 10
Total parameters: 242,762
(32, 10)
[0 0 0 1 3]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1.]
